In [1]:
from dotenv import load_dotenv
import os
import requests

load_dotenv()

API_KEY = os.getenv("TMDB_API_KEY")
API_RA_KEY = os.getenv("TMDB_API_RA_KEY")
headers = {
    "Authorization": f"Bearer {API_RA_KEY}",
    "accept": "application/json"
}

In [6]:

url = "https://api.themoviedb.org/3/search/movie"
params = {
    "query": "Project Hail Mary"
}
response = requests.get(url, headers=headers, params=params)
data = response.json()

data['results'][0].keys()


dict_keys(['adult', 'backdrop_path', 'genre_ids', 'id', 'title', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'release_date', 'softcore', 'video', 'vote_average', 'vote_count'])

In [5]:
url = "https://api.themoviedb.org/3/search/tv"
params = {
    "query": "The Witcher"
}
response = requests.get(url, headers=headers, params=params)
data = response.json()

data['results'][0].keys()

dict_keys(['adult', 'backdrop_path', 'genre_ids', 'id', 'origin_country', 'original_language', 'original_name', 'overview', 'popularity', 'poster_path', 'first_air_date', 'softcore', 'name', 'vote_average', 'vote_count'])

In [92]:
url = "https://api.themoviedb.org/3/trending/all/week"
poster_base_url = "https://image.tmdb.org/t/p/w500"
params = {}

response = requests.get(url, headers=headers, params=params)
data = response.json()

for r in data['results']:
    name = r['original_title'] if r['media_type'] == 'movie' else r['original_name']
    print(name)
    #print(r['poster_path'])
    poster_path = r['poster_path']
    if poster_path:
        poster_url = poster_base_url + poster_path
        response = requests.get(poster_url)
        with open(f"temp/posters/{name}.jpg", "wb") as f:
            f.write(response.content)

Spider-Noir
Backrooms
Obsession
Scary Movie
Hokum
Euphoria
Masters of the Universe
In the Grey
Disclosure Day
FROM
The Boys
Hoppers
Toy Story 5
Iron Lung
Michael
Project Hail Mary
Widow's Bay
Star City
Off Campus
The Legend of Vox Machina


In [80]:
from bs4 import BeautifulSoup
import re

QUALITY_REGEX = r"(2160p|1080p|720p|480p)"
SE_REGEX = r"S(\d{1,2})E(\d{1,2})"

JUNK = [
    "WEBRip", "WEB-DL", "WEB", "AMZN", "HDRip", "BluRay", "BRRip",
    "x264", "x265", "HEVC", "10Bit", "DDP5", "DDP5.1", "AAC",
    "Atmos", "H264", "H.264", "FLUX", "ETHEL", "MeGusta",
    "NeoNoir", "BONE", "RMTeam", "YIFY", "CAM", "PROPER",
    "REPACK", "DSNP", "DCPRip", "DCPRIP"
]

def clean_title(name: str):
    name = re.sub(r"\[.*?\]|\(.*?\)", "", name)
    name = name.replace(".", " ").replace("_", " ")

    for j in JUNK:
        name = name.replace(j, "")

    name = re.sub(r"\s+", " ", name).strip()
    return name


def parse_release(title: str):
    # QUALITY
    quality_match = re.search(QUALITY_REGEX, title)
    quality = quality_match.group(1) if quality_match else None

    # SEASON / EPISODE
    se_match = re.search(SE_REGEX, title, re.IGNORECASE)
    if se_match:
        season_episode = f"S{int(se_match.group(1)):02d}E{int(se_match.group(2)):02d}"
    else:
        season_episode = None

    # CLEAN NAME
    name = clean_title(title)

    return {
        "name": name,
        "season_episode": season_episode,
        "quality": quality
    }

url = "https://thepibay.site/search/Project%20Hail%20Mary"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

items = []

for row in soup.find_all("tr"):
    title_tag = row.select_one("a.detLink")
    magnet_tag = row.find("a", href=lambda x: x and x.startswith("magnet:"))

    if not title_tag:
        continue

    raw_title = title_tag.text.strip()
    page_link = title_tag["href"]
    magnet = magnet_tag["href"] if magnet_tag else None

    parsed = parse_release(raw_title)

    items.append({
        "raw": raw_title,
        "quality": parsed["quality"],
        "magnet": magnet,
        "page_link": page_link
    })
items

[{'raw': 'Project Hail Mary (2026) [1080p] [WEBRip] [5.1]',
  'quality': '1080p',
  'magnet': 'magnet:?xt=urn:btih:3F2F600C7A5637DE5ADF972B053996E57F2B8B0D&dn=Project+Hail+Mary+%282026%29+%5B1080p%5D+%5BWEBRip%5D+%5B5.1%5D&tr=http%3A%2F%2Fp4p.arenabg.com%3A1337%2Fannounce&tr=udp%3A%2F%2F47.ip-51-68-199.eu%3A6969%2Fannounce&tr=udp%3A%2F%2F9.rarbg.me%3A2780%2Fannounce&tr=udp%3A%2F%2F9.rarbg.to%3A2710%2Fannounce&tr=udp%3A%2F%2F9.rarbg.to%3A2730%2Fannounce&tr=udp%3A%2F%2F9.rarbg.to%3A2920%2Fannounce&tr=udp%3A%2F%2Fopen.stealth.si%3A80%2Fannounce&tr=udp%3A%2F%2Fopentracker.i2p.rocks%3A6969%2Fannounce&tr=udp%3A%2F%2Ftracker.coppersurfer.tk%3A6969%2Fannounce&tr=udp%3A%2F%2Ftracker.cyberia.is%3A6969%2Fannounce&tr=udp%3A%2F%2Ftracker.dler.org%3A6969%2Fannounce&tr=udp%3A%2F%2Ftracker.internetwarriors.net%3A1337%2Fannounce&tr=udp%3A%2F%2Ftracker.leechers-paradise.org%3A6969%2Fannounce&tr=udp%3A%2F%2Ftracker.openbittorrent.com%3A6969%2Fannounce&tr=udp%3A%2F%2Ftracker.opentrackr.org%3A1337&tr=udp%3